# pq-adbc-advisor — starter notebook

Scans the currently-attached Fabric workspace and returns an item-by-item impact report for the ODBC → ADBC migration.

**This is a starting-point diagnostic, not a final audit.** Treat the report as the first pass to find the bulk of the work. For high-value production reports, validate the ADBC path in a copy of the item before flipping production over.

For per-connector dates and the migration timeline, the source of truth is the public migration guide at [aka.ms/adbc-migration](https://aka.ms/adbc-migration).

---

## What this notebook does

1. Installs the `pq-adbc-advisor` package into this notebook session.
2. Runs `scan_workspace()` against the currently-attached workspace.
3. Renders the impact report inline.
4. Saves an HTML copy of the report to the default lakehouse (if one is attached).

Runtime: ~90 seconds on a 200-artifact workspace, longer for larger workspaces.

---

## Step 1 — Install the package

The tool ships as a Python package on PyPI. This installs it into the current notebook session only.

In [ ]:
%pip install pq-adbc-advisor --quiet

## Step 2 — Scan the current workspace

`scan_workspace()` auto-detects the workspace this notebook is attached to. It reads item definitions, walks every M expression it finds, and classifies every migrating connector call by cutover risk. It does not rewrite any M and does not trigger any refreshes.

In [ ]:
from pq_adbc_advisor import scan_workspace

report = scan_workspace()
report

## Step 3 — Save the report

This writes a self-contained HTML file that you can download from the lakehouse Files pane and share inside your org. Excerpts are secret-redacted before rendering, so it's safe to share.

If no default lakehouse is attached, the report is saved to `/tmp/` and the path is printed.

In [ ]:
output_path = report.to_html("/lakehouse/default/Files/pq_adbc_advisor_impact.html")
print(f"Report saved to: {output_path}")

## Optional — Tenant-wide scan (admin only)

If you have Fabric admin permissions and want to inventory every workspace in the tenant, uncomment and run the cell below. Tenant scans use the Fabric admin Scanner API and take 15–30 minutes for a mid-sized tenant.

In [ ]:
# from pq_adbc_advisor import scan_tenant
#
# tenant_report = scan_tenant()
# tenant_report.to_html("/lakehouse/default/Files/pq_adbc_advisor_tenant.html")

## Optional — Opt out of anonymous telemetry

Every scan sends anonymous counts to the Power Query team so we can measure adoption and prioritize connector coverage — SHA-256-hashed tenant and user identifiers only, no raw values by default. To opt out at any time, run the cell below. The opt-out is persisted across kernel restarts.

In [ ]:
# from pq_adbc_advisor import disable_telemetry
# disable_telemetry()

---

## Where to file

- **Bugs and feature requests:** [github.com/microsoft/fabric-toolbox/issues](https://github.com/microsoft/fabric-toolbox/issues) — tag with `pq-adbc-advisor`
- **Corner cases the tool flags but the standard connector doesn't cover:** adbcmigration@microsoft.com
- **Public migration guide:** [aka.ms/adbc-migration](https://aka.ms/adbc-migration)